# Evaluation Dataset

## Libraries

In [ ]:
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
client = OpenAI()

## Data Load

In [ ]:
def load_json_to_dataframe(folder: str) -> pd.DataFrame:
    dfs = []

    for file in Path(folder).glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            raw = f.read().strip()

            try:
                records = json.loads(raw)
            except json.JSONDecodeError:
                records = [json.loads(line) for line in raw.splitlines() if line.strip()]

        if isinstance(records, dict):
            records = [records]

        temp_df = pd.DataFrame(records)

        if temp_df.empty:
            continue

        if "content" in temp_df.columns:
            temp_df["text"] = temp_df["content"]

        # ensure required columns exist
        if "url" not in temp_df.columns or "text" not in temp_df.columns:
            continue

        # restore schema
        temp_df["type"] = temp_df.get("type", "chunked")
        temp_df["depth"] = temp_df.get("depth", None)
        temp_df["file"] = file.stem

        temp_df = temp_df[~temp_df["url"].astype(str).str.contains("qa://", na=False)]

        dfs.append(temp_df[["url", "type", "depth", "text", "file"]])

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [26]:
data_bc = load_json_to_dataframe("/Users/antoniooliveira/Documents/GitHub/IAPMEI-chatbot-v3/data/01_extracted")
data_ac = load_json_to_dataframe("/Users/antoniooliveira/Documents/GitHub/IAPMEI-chatbot-v3/data/03_chunked/c600_60")

In [27]:
data_bc.head()

,url,type,depth,text,file
0,https://algarve.portugal2030.pt/,html,0,Programas do Portugal 2030 PESSOAS 2030 COMPET...,algarve2030
1,https://algarve.portugal2030.pt/ajuda-artigos/,html,1,Programas do Portugal 2030 PESSOAS 2030 COMPET...,algarve2030
2,https://algarve.portugal2030.pt/eventos/,html,1,Programas do Portugal 2030 PESSOAS 2030 COMPET...,algarve2030
3,https://algarve.portugal2030.pt/noticias/,html,1,Programas do Portugal 2030 PESSOAS 2030 COMPET...,algarve2030
4,https://algarve.portugal2030.pt/o-portugal-2030/,html,1,Programas do Portugal 2030 PESSOAS 2030 COMPET...,algarve2030


In [28]:
data_ac.head()

,url,type,depth,text,file
0,https://algarve.portugal2030.pt/,chunked,None,Fonte: algarve.portugal2030.pt: Algarve 2030 A...,algarve2030
1,https://algarve.portugal2030.pt/,chunked,None,Fonte: algarve.portugal2030.pt: . Saiba mais E...,algarve2030
2,https://algarve.portugal2030.pt/,chunked,None,Fonte: algarve.portugal2030.pt: . “A água é um...,algarve2030
3,https://algarve.portugal2030.pt/ajuda-artigos/,chunked,None,Fonte: Ajuda artigos: Ajuda Precisa de ajuda? ...,algarve2030
4,https://algarve.portugal2030.pt/ajuda-artigos/,chunked,None,Fonte: Ajuda artigos: . Limpar filtros Ordenar...,algarve2030


# Evaluation Dataset Creation

In [29]:
dfs = {
    file_name: df_file.copy()
    for file_name, df_file in data_ac.groupby("file")
}

In [30]:
dfs.keys()

dict_keys(['alentejo2030', 'algarve2030', 'centro2030', 'compete2030', 'iapmei_website', 'lisboa2030', 'norte2030', 'portugal2030'])

In [27]:
def generate_qa_from_df(
    df: pd.DataFrame,
    text_column: str,
    model: str = "gpt-4o-mini",
    n_questions: int = 5,
    max_chars: int = 6000
):
    """
    Receives a DataFrame with text and returns n question–answer pairs in JSON format.
    """

    # 1. Prepare text (truncate to avoid context overflow)
    text = "\n\n".join(df[text_column].dropna().astype(str).tolist())
    text = text[:max_chars]

    # 2. Prompt
    prompt = f"""
        És um assistente que cria perguntas e respostas com base na informação fornecida. 
        Sê sempre claro na questão que colocas, garantindo sempre que tem informação suficiente 
        para que seja possível obter uma resposta. Lembra-te de que, no futuro, a busca pela
        resposta será feita relativamente a todos os programas.
        
        Regras:
        - Gera exatamente {n_questions} perguntas e respostas
        - As perguntas devem ser claras e relevantes
        - As respostas devem basear-se apenas na informação disponível e ser o mais completas possível
        - Responde APENAS em JSON válido sempre, independentemente de tudo o resto.
        
        Texto:
        \"\"\"
        {text}
        \"\"\"
        """

    # 3. Call model
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.2,
        response_format={"type": "json_object"}
    )


    # 4. Parse JSON safely
    content = response.choices[0].message.content.strip()

    try:
        qa = json.loads(content)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model did not return valid JSON:\n{content}") from e

    return qa


In [28]:
all_qa = {}

for file_name, df in dfs.items():
    qa_pairs = generate_qa_from_df(
        df=df,
        text_column="content"
    )
    all_qa[file_name] = qa_pairs

In [12]:
[q["questions"] for q in all_qa["portugal2030"]["questions"]]

['O que é o Portugal 2030 e qual é o seu objetivo?',
 'Quantos milhões de euros já foram pagos aos beneficiários do Portugal 2030?',
 'Quando e onde ocorreu a Mostra dos Fundos Europeus?',
 'Quais são os programas que compõem o Portugal 2030?',
 'Como os interessados podem entrar em contato com a Linha dos Fundos para esclarecer dúvidas?']

In [29]:
all_qa

{'alentejo2030': {'perguntas_e_respostas': [{'pergunta': 'O que é o programa Alentejo 2030?',
    'resposta': 'O Alentejo 2030 é um programa que visa transformar vidas na região do Alentejo, impulsionando a inovação, a sustentabilidade e o crescimento. O programa também oferece oportunidades para beneficiar dos Fundos Europeus disponíveis.'},
   {'pergunta': 'Quais são algumas das novas prioridades específicas integradas na reprogramação do Alentejo 2030?',
    'resposta': 'As novas prioridades incluem a Defesa, a Água, a Habitação a preços acessíveis e sustentável, e as Competências para a descarbonização. Estas áreas focam em tecnologias de dupla utilização, gestão sustentável da água, acesso à habitação e educação voltada para a transição climática.'},
   {'pergunta': 'Quando foi aprovada a reprogramação do Alentejo 2030 pela Comissão Europeia?',
    'resposta': 'A reprogramação do Alentejo 2030 foi aprovada pela Comissão Europeia no dia 15 de dezembro de 2025.'},
   {'pergunta': 'Q

In [ ]:
with open("evaluation/evaluation_dataset_v4.json", "w", encoding="utf-8") as f:
    json.dump(
        all_qa,
        f,
        ensure_ascii=False, 
        indent=2             
    )
